# Meshes

**Geometry type:** `mesh`

Triangulated 3-D surface meshes — cell boundaries from EM segmentation,
brain surfaces from MRI, organelle hulls.

1. Generate a synthetic closed surface (sphere approximation)
2. Write with per-vertex attributes
3. Open lazily and inspect
4. Read all geometry
5. Spatial bounding-box query
6. Multi-mesh store (cell segmentation, via `object_ids`)
7. OBJ round-trip
8. Validate

In [2]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.meshes import write_mesh, read_mesh
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE       = os.path.join(_tmpdir, "brain_surface.zarrvectors")
STORE_MULTI = os.path.join(_tmpdir, "cells.zarrvectors")
OBJ_PATH    = os.path.join(_tmpdir, "surface.obj")
OBJ_OUT     = os.path.join(_tmpdir, "surface_out.obj")
print("Stores:", STORE, STORE_MULTI)

Stores: /tmp/zvf_examples_etamp05c/brain_surface.zarrvectors /tmp/zvf_examples_etamp05c/cells.zarrvectors


## 1 · Generate a synthetic closed surface

In [3]:
def uv_sphere(radius, lat_steps=40, lon_steps=80, offset=(0., 0., 0.)):
    """Generate a UV-sphere with positions, faces, normals, curvature."""
    verts, norms = [], []
    for i in range(lat_steps + 1):
        theta = np.pi * i / lat_steps
        for j in range(lon_steps):
            phi = 2 * np.pi * j / lon_steps
            x = radius * np.sin(theta) * np.cos(phi)
            y = radius * np.sin(theta) * np.sin(phi)
            z = radius * np.cos(theta)
            verts.append([x + offset[0], y + offset[1], z + offset[2]])
            norms.append([np.sin(theta)*np.cos(phi),
                          np.sin(theta)*np.sin(phi),
                          np.cos(theta)])
    verts = np.array(verts, dtype=np.float32)
    norms = np.array(norms, dtype=np.float32)

    faces = []
    for i in range(lat_steps):
        for j in range(lon_steps):
            a = i * lon_steps + j
            b = a + lon_steps
            c = a + 1 if j < lon_steps - 1 else i * lon_steps
            d = b + 1 if j < lon_steps - 1 else (i + 1) * lon_steps
            if i > 0:
                faces.append([a, b, c])
            if i < lat_steps - 1:
                faces.append([b, d, c])
    faces = np.array(faces, dtype=np.int32)

    # Synthetic curvature = 1/radius (constant for sphere, scaled by noise)
    rng_c = np.random.default_rng(42)
    curvature = np.full(len(verts), 1.0 / radius, dtype=np.float32)
    curvature += rng_c.normal(0, 0.005, len(verts)).astype(np.float32)

    return verts, faces, norms, curvature

# Brain-surface-like hemisphere: 150 mm radius centred at (150, 150, 150)
vertices, faces, normals, curvature = uv_sphere(
    radius=50., lat_steps=50, lon_steps=100,
    offset=(150., 150., 150.)
)
print(f"vertices  : {vertices.shape}")
print(f"faces     : {faces.shape}")
print(f"normals   : {normals.shape}")
print(f"curvature : [{curvature.min():.4f}, {curvature.max():.4f}]")


vertices  : (5100, 3)
faces     : (9800, 3)
normals   : (5100, 3)
curvature : [0.0018, 0.0373]


## 2 · Write with per-vertex attributes

In [4]:
write_mesh(
    STORE,
    vertices=vertices,
    faces=faces,
    chunk_shape=(50.0, 50.0, 50.0),
    bin_shape=(12.5, 12.5, 12.5),
    vertex_attributes={
        "normal":    normals,    # (N, 3) vector
        "curvature": curvature,  # (N,)
    },
)
print("Write complete.")

Write complete.


## 3 · Open lazily and inspect

In [5]:
store = open_zvr(STORE)
print(f"geometry_types: {store.geometry_types}")
print(f"ndim          : {store.ndim}")
print(f"n_objects     : {store[0].object_index.object_count}  (1 mesh surface)")
print(f"vertex_count  : {store[0].vertex_count:,}")
lo, hi = np.array(store.bounds[0]), np.array(store.bounds[1])
print(f"bounds        : lo={lo.round(1)}  hi={hi.round(1)}")

geometry_types: ['mesh']
ndim          : 3
n_objects     : 1  (1 mesh surface)
vertex_count  : 5,100
bounds        : lo=[100. 100. 100.]  hi=[200. 200. 200.]


## 4 · Read all geometry

In [6]:
result = read_mesh(STORE)
print(f"vertex_count : {result['vertex_count']:,}")
print(f"face_count   : {result['face_count']:,}")
print(f"vertices     : {result['vertices'].shape}")
print(f"faces        : {result['faces'].shape}   (remapped to output vertex order)")

# Spot-check face winding (cross product should point outward from sphere centre)
v = result['vertices']
f = result['faces']
centre = np.array([150., 150., 150.])
dots = []
for face_row in f[:5]:
    a, b, c = v[face_row[0]], v[face_row[1]], v[face_row[2]]
    n = np.cross(b - a, c - a)
    centroid = (a + b + c) / 3
    dots.append(float(np.dot(n, centroid - centre)))
print(f"\nFace normal · outward direction (first 5): {[f'{x:.1f}' for x in dots]}")
print("  (positive = outward-facing ✓)")

vertex_count : 5,100
face_count   : 9,046
vertices     : (5100, 3)
faces        : (9046, 3)   (remapped to output vertex order)

Face normal · outward direction (first 5): ['491.9', '488.9', '491.9', '488.9', '491.9']
  (positive = outward-facing ✓)


## 5 · Spatial bounding-box query

In [7]:
lo = np.array([100., 100., 100.])
hi = np.array([200., 200., 200.])

bbox_result = read_mesh(STORE, bbox=(lo, hi))
print(f"Faces (centroid in bbox) : {bbox_result['face_count']:,}")
print(f"Vertices in result       : {bbox_result['vertex_count']:,}")
print("  (vertices may extend outside bbox — face centroid is the assignment criterion)")

Faces (centroid in bbox) : 9,046
Vertices in result       : 5,100
  (vertices may extend outside bbox — face centroid is the assignment criterion)


## 6 · Multi-mesh store (cell segmentation)

In [8]:
# Generate 20 small sphere-shaped cells at random positions
rng = np.random.default_rng(5)
n_cells = 20
verts_list, faces_list = [], []
cell_volumes = []
cell_types   = rng.integers(0, 3, n_cells).astype(np.int32)

for i in range(n_cells):
    radius = rng.uniform(8, 20)
    cx, cy, cz = rng.uniform(50, 950, 3)
    v, f, _, _ = uv_sphere(radius, lat_steps=15, lon_steps=30, offset=(cx, cy, cz))
    verts_list.append(v)
    faces_list.append(f)
    cell_volumes.append(float((4/3) * np.pi * radius**3))

# Combine into a single store and tag each vertex with its cell ID
vertices_multi = np.concatenate(verts_list, axis=0).astype(np.float32)
offsets = np.cumsum([0] + [len(v) for v in verts_list[:-1]])
faces_multi = np.concatenate(
    [f + off for f, off in zip(faces_list, offsets)], axis=0
).astype(np.int32)
object_ids_multi = np.concatenate(
    [np.full(len(v), i, dtype=np.int32) for i, v in enumerate(verts_list)]
)

write_mesh(
    STORE_MULTI,
    vertices=vertices_multi,
    faces=faces_multi,
    chunk_shape=(100., 100., 100.),
    bin_shape=(25., 25., 25.),
    object_ids=object_ids_multi,
)
store_m = open_zvr(STORE_MULTI)
print(f"Multi-mesh store: {store_m[0].object_index.object_count} cells, "
      f"{store_m[0].vertex_count:,} total vertices")

Multi-mesh store: 20 cells, 9,600 total vertices


## 6b · Spatial query on the multi-mesh store

In [9]:
# Spatial query — bbox filtering on faces works correctly (face centroid in bbox)
lo_q, hi_q = np.array([200., 200., 200.]), np.array([600., 600., 600.])
bbox_m = read_mesh(STORE_MULTI, bbox=(lo_q, hi_q))
print(f"Faces with centroid in region: {bbox_m['face_count']:,}")
print(f"Vertices in region result    : {bbox_m['vertex_count']:,}")

# NOTE: read_mesh(object_ids=[k]) currently returns the full store — the
# object-ID filter is not yet implemented for meshes. Until it is, use
# bbox queries or split each cell into its own store.

Faces with centroid in region: 0
Vertices in region result    : 0


## 7 · OBJ round-trip

In [10]:
# Write a small OBJ from the sphere vertices/faces
with open(OBJ_PATH, "w") as f:
    f.write("# zarr-vectors example sphere\n")
    for v in vertices[:200]:      # first 200 vertices for brevity
        f.write(f"v {v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n")
    for face in faces[:100]:
        # OBJ uses 1-based indices; skip faces that reference vertices > 200
        if face.max() < 200:
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")
print(f"Wrote test OBJ: {OBJ_PATH}")

# Ingest OBJ → ZVF
STORE_OBJ = os.path.join(_tmpdir, "from_obj.zarrvectors")
from zarr_vectors.ingest.obj import ingest_obj
ingest_obj(OBJ_PATH, STORE_OBJ, chunk_shape=(50., 50., 50.))
r_obj = read_mesh(STORE_OBJ)
print(f"Ingested OBJ: {r_obj['vertex_count']} vertices, {r_obj['face_count']} faces")

# Export back to OBJ
from zarr_vectors.export.obj import export_obj
export_obj(STORE_OBJ, OBJ_OUT)
print(f"Exported to {OBJ_OUT}")


Wrote test OBJ: /tmp/zvf_examples_etamp05c/surface.obj
Ingested OBJ: 200 vertices, 0 faces
Exported to /tmp/zvf_examples_etamp05c/surface_out.obj


## 8 · Validate

In [11]:
rv = validate(STORE, level=3)
print(rv.summary())

rv_m = validate(STORE_MULTI, level=3)
print(rv_m.summary())

Level 3 validation: PASS
  25 passed, 0 warnings, 0 errors
Level 3 validation: PASS
  24 passed, 0 warnings, 0 errors


## Summary

| Step | API |
|------|-----|
| Write single | `write_mesh(path, vertices, faces, vertex_attributes, chunk_shape, bin_shape)` |
| Write multi | `write_mesh(path, vertices, faces, object_ids=...)` |
| Read | `read_mesh(path)` → `{vertices, faces, vertex_count, face_count}` |
| Bbox query | `read_mesh(path, bbox=(lo, hi))` (filters by face centroid) |
| OBJ ingest | `ingest_obj(obj_path, store_path, chunk_shape)` |
| OBJ export | `export_obj(store_path, obj_path)` |

> **Known limitation:** `read_mesh(object_ids=[k])` currently does not filter
> — it returns the full store. Use bbox queries until the object-ID filter
> is implemented.